# Day 10 — Summarization with HuggingFace

**Roadmap Phase:** Phase 2 — Applied NLP  
**Objective:** Build abstractive and extractive summarization pipelines, understand the difference, and probe length/quality tradeoffs  
**Commit target:** `feat: day10 - summarization pipelines`

---

### Day 9 → Day 10 shift
| | Day 9 (QA) | Day 10 (Summarization) |
|---|---|---|
| Task type | Extractive — finds a span in the context | Abstractive — generates new text |
| Model output | A slice of the input | New sentences not in the original |
| Key metric | Confidence score | Summary length, coherence |

### 15-minute rule
If anything blocks you for >15 min, write it down and move on.

## 0. Setup & Imports

In [ ]:
# Install if needed
# !pip install transformers torch pandas rouge-score

In [ ]:
from transformers import pipeline
import pandas as pd

print("Imports OK")

## 1. Load the Summarization Pipeline

`facebook/bart-large-cnn` is fine-tuned on CNN/DailyMail news articles.  
It's the standard go-to for abstractive summarization — fast, reliable, and well-documented.

In [ ]:
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn"
)

print("Model loaded:", summarizer.model.config.model_type)

## 2. Source Documents

Use varied domains and lengths so you can observe how the model handles each.

In [ ]:
documents = {
    "transformers_history": """
        The Transformer architecture was introduced in the landmark 2017 paper 'Attention Is All You Need'
        by Vaswani et al. at Google Brain. Prior to this, sequence modelling relied heavily on recurrent
        neural networks (RNNs) and LSTMs, which processed tokens sequentially and struggled with long-range
        dependencies due to vanishing gradients. The Transformer replaced recurrence entirely with
        self-attention mechanisms, allowing all tokens in a sequence to attend to each other in parallel.
        This parallelism made training significantly faster and enabled scaling to much larger datasets.
        BERT, released by Google in 2018, applied the Transformer encoder for bidirectional language
        understanding. GPT, released by OpenAI the same year, used the decoder for autoregressive
        text generation. These two lines of research have dominated NLP ever since, leading to models
        like GPT-4, Claude, Gemini, and LLaMA.
    """,

    "rag_overview": """
        Retrieval-Augmented Generation, commonly known as RAG, is a technique that enhances large
        language models by grounding their responses in external, retrieved knowledge. A standard RAG
        pipeline consists of two main components: a retriever and a generator. The retriever searches
        a document store — typically using dense vector embeddings and approximate nearest-neighbour
        search (e.g., FAISS) — to find chunks of text that are semantically relevant to a query.
        These retrieved chunks are then concatenated with the user's question and passed as context
        to the language model, which generates a final answer grounded in the retrieved evidence.
        RAG significantly reduces hallucination compared to relying on the model's parametric knowledge
        alone. It also allows knowledge to be updated without retraining the model — simply update
        the document store. RAG is widely used in enterprise chatbots, customer support systems,
        and any application where factual accuracy over a private knowledge base is critical.
    """,

    "nepal_short": """
        Nepal is a small, landlocked country in South Asia bordered by China to the north and India
        to the south. Despite its small size, it contains some of the most dramatic geography on Earth,
        including eight of the world's ten highest peaks. Mount Everest, the tallest at 8,848 metres,
        sits on the Nepal-China border. The capital Kathmandu lies in a valley at roughly 1,400 metres
        above sea level and is the cultural and economic heart of the country.
    """,

    "llm_engineering": """
        Large language model engineering is an emerging discipline that bridges software engineering,
        machine learning, and product development. LLM engineers are responsible for building systems
        that leverage pre-trained foundation models — typically through APIs or local inference —
        to solve real-world tasks. Core skills include prompt engineering, fine-tuning on domain data,
        building retrieval-augmented generation pipelines, evaluating model outputs systematically,
        and deploying models reliably at scale. Unlike traditional ML engineering, which focuses on
        training models from scratch, LLM engineering emphasises composing and adapting existing
        powerful models efficiently. The field is evolving rapidly, with new techniques such as
        chain-of-thought prompting, tool use, and agentic workflows becoming standard practice.
        Companies across industries are investing heavily in LLM engineering talent as they seek
        to integrate AI into their core products.
    """,
}

print("Documents defined:", list(documents.keys()))
for k, v in documents.items():
    print(f"  {k}: {len(v.split())} words")

## 3. Run Summarization — Default Settings

In [ ]:
results_default = []

for doc_name, text in documents.items():
    summary = summarizer(
        text.strip(),
        max_length=80,
        min_length=20,
        do_sample=False        # greedy / beam search
    )[0]["summary_text"]

    results_default.append({
        "document":      doc_name,
        "source_words":  len(text.split()),
        "summary":       summary,
        "summary_words": len(summary.split()),
        "compression":   f"{(1 - len(summary.split()) / len(text.split())) * 100:.0f}%"
    })

df_default = pd.DataFrame(results_default)
print("Done.")

In [ ]:
pd.set_option("display.max_colwidth", 100)
df_default[["document", "source_words", "summary_words", "compression", "summary"]]

## 4. Length Parameter Sweep

See how `max_length` affects the output quality.  
Run the same document through three length configs and compare.

In [ ]:
target_doc = "rag_overview"   # change to any key in documents
text = documents[target_doc].strip()

length_configs = [
    {"max_length": 30,  "min_length": 10,  "label": "very_short"},
    {"max_length": 80,  "min_length": 20,  "label": "medium"},
    {"max_length": 150, "min_length": 60,  "label": "long"},
]

sweep_results = []
for cfg in length_configs:
    out = summarizer(text, max_length=cfg["max_length"], min_length=cfg["min_length"], do_sample=False)[0]
    sweep_results.append({
        "config":        cfg["label"],
        "max_length":    cfg["max_length"],
        "actual_words":  len(out["summary_text"].split()),
        "summary":       out["summary_text"],
    })

df_sweep = pd.DataFrame(sweep_results)
df_sweep[["config", "max_length", "actual_words", "summary"]]

## 5. Sampling vs Beam Search

`do_sample=False` uses beam search → deterministic, more factual  
`do_sample=True` uses sampling → varied, sometimes more fluent but less reliable

Run both on the same document and compare.

In [ ]:
text = documents["llm_engineering"].strip()

beam_summary   = summarizer(text, max_length=80, min_length=20, do_sample=False)[0]["summary_text"]
sample_summary = summarizer(text, max_length=80, min_length=20, do_sample=True,
                             temperature=0.9, top_p=0.95)[0]["summary_text"]

print("=== BEAM SEARCH ===")
print(beam_summary)
print(f"\n({len(beam_summary.split())} words)")

print("\n=== SAMPLING (temp=0.9) ===")
print(sample_summary)
print(f"\n({len(sample_summary.split())} words)")

## 6. Edge Case — Very Short Input

What happens when the input is shorter than `min_length`?

In [ ]:
short_text = "RAG combines retrieval with generation to reduce hallucination."

try:
    out = summarizer(short_text, max_length=80, min_length=20, do_sample=False)
    print("Output:", out[0]["summary_text"])
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

## 7. Interactive Cell

Paste any text you want summarised — article, paper abstract, README, etc.

In [ ]:
# ── Edit and re-run ────────────────────────────────────────────────────────
MY_TEXT = """
    Paste your text here. It should be at least 50 words for meaningful summarization.
    The model works best on factual, informational text like news articles,
    Wikipedia paragraphs, documentation, or research abstracts.
"""

MY_MAX_LENGTH = 80
MY_MIN_LENGTH = 20
# ──────────────────────────────────────────────────────────────────────────

out = summarizer(MY_TEXT.strip(), max_length=MY_MAX_LENGTH, min_length=MY_MIN_LENGTH, do_sample=False)
print("Summary:", out[0]["summary_text"])
print(f"\n{len(MY_TEXT.split())} words → {len(out[0]['summary_text'].split())} words")

## 8. Daily Log

Fill in before committing.

```
Roadmap phase   : Phase 2 — Applied NLP
Objective       : Abstractive summarization with BART
What was built  : 
Struggles       : 
Next task       : Day 11 — Embeddings (sentence-transformers)
Energy level    : /5
15-min rule     : triggered? Y/N — what was the blocker?
One blocker     : 
```

---

## Key Concepts Recap

| Concept | What it means |
|---------|---------------|
| Abstractive summarization | Generates new sentences — not just extracting spans |
| `max_length` / `min_length` | Token bounds on the generated summary |
| `do_sample=False` | Beam search — deterministic and factual |
| `do_sample=True` | Sampling — varied output, less reliable for facts |
| Compression ratio | `(1 - summary_words / source_words) × 100%` |
| Next step | Day 11: Embeddings — map text to dense vectors for similarity/retrieval |